In [3]:
import time
import numpy as np
import pandas as pd
import yfinance as yf
from datetime import datetime, timedelta
from scipy.optimize import minimize


# 0) Parameter Settings

ann_fac = 252

# Scoring function: dual standardization
# Sharpe weight = 0.7, Alpha weight = 0.3, Beta penalty
wS, wA = 0.7, 0.3
lam = 0.2  # beta penalty strength

# Attack sector definition and allocation constraint
attack_sectors = {"Semiconductors", "Industrials", "Energy"}
attack_target = 0.70  # 70% in attack sectors, 30% in others

# Minimum weight per asset
min_w = 0.05

# Maximum number of stocks per sector
max_per_sector = 2

benchmark = "SPY"

# Last 3 months window for scoring
end_3m = datetime.today()
start_3m = end_3m - timedelta(days=92)

# Last 10 years window for portfolio optimization
end_10y = datetime.today()
start_10y = end_10y - timedelta(days=365 * 10 + 10)


# 1) Investment Universe
# (Modify according to course requirements if needed)

universe = {
    "Semiconductors": ["NVDA", "TSM", "ASML", "AVGO"],
    "Industrials": ["CAT", "RTX", "GE", "MMM"],
    "Energy": ["XOM", "CVX", "COP", "SHEL"],
    "Materials": ["BHP", "LIN", "NEM", "APD"],
    "ConsumerStaples": ["PG", "PEP", "WMT", "COST"],
    "Healthcare": ["JNJ", "ABBV", "LLY", "UNH"],
}

sector_of = {t: sec for sec, lst in universe.items() for t in lst}
all_candidates = sorted(list(sector_of.keys()))
download_list_3m = sorted(set(all_candidates + [benchmark]))


# 2) Helper Functions

def zscore(s: pd.Series) -> pd.Series:
    s = s.astype(float)
    return (s - s.mean()) / s.std(ddof=0)

def safe_download_close(tickers, start, end, pause_sec=0.2):
    """
    Robust data download:
    auto_adjust=True ensures Close is adjusted for splits/dividends
    """
    data = yf.download(
        tickers,
        start=start,
        end=end,
        auto_adjust=True,
        progress=False,
        group_by="column"
    )

    # Handle MultiIndex columns
    if isinstance(data.columns, pd.MultiIndex):
        if "Close" not in data.columns.get_level_values(0):
            raise RuntimeError("No Close field returned.")
        px = data["Close"].copy()
    else:
        if "Close" not in data.columns:
            raise RuntimeError("No Close column returned.")
        px = data["Close"].copy()

    px = px.dropna(how="all").dropna(axis=1, how="all")

    time.sleep(pause_sec)
    return px

def get_rf_dtb3_avg(sample_start, sample_end):
    """
    Fetch risk-free rate from FRED (DTB3: 3-Month Treasury Bill)
    Returns annualized rf in decimal form
    """
    fred = pd.read_csv("https://fred.stlouisfed.org/graph/fredgraph.csv?id=DTB3")
    date_col = fred.columns[0]
    rate_col = fred.columns[1]

    fred[date_col] = pd.to_datetime(fred[date_col], errors="coerce")
    fred[rate_col] = pd.to_numeric(fred[rate_col], errors="coerce")
    fred = fred.dropna(subset=[date_col, rate_col])

    samp = fred[(fred[date_col] >= sample_start) & (fred[date_col] <= sample_end)]
    if samp.empty:
        raise RuntimeError("DTB3 empty after filtering window.")

    return float(samp[rate_col].mean() / 100.0)


# 3) 3-Month Scoring: Sharpe / Alpha / Beta

px_3m = safe_download_close(download_list_3m, start_3m, end_3m)

# Log returns
ret_3m = np.log(px_3m).diff().dropna()

if benchmark not in ret_3m.columns:
    raise RuntimeError("SPY not downloaded successfully for scoring.")

rm = ret_3m[benchmark].dropna()

rows = []
for t in all_candidates:
    if t not in ret_3m.columns:
        continue

    ri = ret_3m[t].dropna()

    df = pd.concat([ri, rm], axis=1, join="inner")
    df.columns = ["Ri", "Rm"]

    if len(df) < 30:
        continue

    # Annualized Sharpe ratio
    sharpe_ann = (df["Ri"].mean() / df["Ri"].std(ddof=1)) * np.sqrt(ann_fac)

    # CAPM regression: Ri = alpha + beta * Rm
    X = np.vstack([np.ones(len(df)), df["Rm"].values]).T
    y = df["Ri"].values

    alpha_daily, beta = np.linalg.lstsq(X, y, rcond=None)[0]
    alpha_ann = alpha_daily * ann_fac

    rows.append({
        "Sector": sector_of[t],
        "Ticker": t,
        "Nobs": len(df),
        "Sharpe_Ann": sharpe_ann,
        "Alpha_Ann": alpha_ann,
        "Beta": beta,
        "AbsBeta": abs(beta),
    })

score_df = pd.DataFrame(rows).dropna()

# Dual standardization + beta penalty
score_df["z_Sharpe"] = zscore(score_df["Sharpe_Ann"])
score_df["z_Alpha"] = zscore(score_df["Alpha_Ann"])
score_df["z_AbsBeta"] = zscore(score_df["AbsBeta"])

score_df["Score"] = wS * score_df["z_Sharpe"] + wA * score_df["z_Alpha"] - lam * score_df["z_AbsBeta"]
score_df = score_df.sort_values("Score", ascending=False).reset_index(drop=True)


# 4) Stock Selection

selected = []
sector_count = {}

def can_add(ticker, sector):
    return sector_count.get(sector, 0) < max_per_sector and ticker not in selected

# Select from attack sectors first
for _, r in score_df.iterrows():
    sec = r["Sector"]
    tic = r["Ticker"]

    if sec in attack_sectors and can_add(tic, sec):
        selected.append(tic)
        sector_count[sec] = sector_count.get(sec, 0) + 1

    if len(selected) >= 6:
        break

# Fill remaining slots
for _, r in score_df.iterrows():
    if len(selected) >= 10:
        break

    sec = r["Sector"]
    tic = r["Ticker"]

    if sec not in attack_sectors and can_add(tic, sec):
        selected.append(tic)
        sector_count[sec] = sector_count.get(sec, 0) + 1

# Output selection
sel_df = score_df[score_df["Ticker"].isin(selected)].copy()
sel_df = sel_df.set_index("Ticker").loc[selected].reset_index()

print("\n=== Selected 10 stocks by Score (sector<=2; attack prioritized) ===")
print(sel_df[["Sector","Ticker","Score","Sharpe_Ann","Alpha_Ann","Beta"]].to_string(index=False))


# 5) Markowitz Optimization (10Y)

px_10y = safe_download_close(selected, start_10y, end_10y)

ret_10y = np.log(px_10y).diff().dropna()

mu = ret_10y.mean().values * ann_fac
Sigma = ret_10y.cov().values * ann_fac

rf = get_rf_dtb3_avg(ret_10y.index.min(), ret_10y.index.max())
print(f"\nrf (DTB3 avg over sample) = {rf:.4%}")

cols = list(px_10y.columns)
n = len(cols)

is_attack = np.array([sector_of[t] in attack_sectors for t in cols], dtype=float)

def port_stats(w):
    r = w @ mu
    vol = np.sqrt(w @ Sigma @ w)
    sharpe = (r - rf) / vol if vol > 0 else -1e9
    return r, vol, sharpe

def neg_sharpe(w):
    return -port_stats(w)[2]

constraints = [
    {"type": "eq", "fun": lambda w: np.sum(w) - 1.0},
    {"type": "eq", "fun": lambda w: (w @ is_attack) - attack_target},
]

bounds = [(min_w, 1.0) for _ in range(n)]

# Initial feasible weights
w0 = np.ones(n) / n

res = minimize(neg_sharpe, w0, method="SLSQP", bounds=bounds, constraints=constraints)

w_star = res.x
r, vol, sh = port_stats(w_star)

out = pd.DataFrame({
    "Sector": [sector_of[t] for t in cols],
    "Ticker": cols,
    "Weight": w_star
}).sort_values("Weight", ascending=False)

print("\n=== Max Sharpe Portfolio ===")
print(f"Return={r:.4f}, Vol={vol:.4f}, Sharpe={sh:.4f}")
print(out.round(4).to_string(index=False))


=== Selected 10 stocks by Score (sector<=2; attack=Semis/Industrials/Energy prioritized) ===
         Sector Ticker     Score  Sharpe_Ann  Alpha_Ann      Beta
         Energy    XOM  1.598881    4.021431   1.069512  0.044468
         Energy    CVX  1.541981    4.087959   0.952936  0.099090
    Industrials    RTX  0.797710    2.977228   0.736625  0.622913
 Semiconductors    TSM  0.172934    2.250953   0.804485  2.071298
 Semiconductors   ASML  0.060658    2.070808   0.883031  2.412886
    Industrials    CAT -0.088754    1.866000   0.700085  2.192784
      Materials    LIN  1.444311    4.163837   0.801932  0.250157
     Healthcare    JNJ  1.361459    4.061586   0.716418 -0.189230
      Materials    BHP  0.845961    3.010523   1.032660  1.242838
ConsumerStaples   COST  0.206585    1.844485   0.382348  0.122460

rf (DTB3 avg over 10Y sample) = 2.2284%

=== Max Sharpe Portfolio (10Y, rf=DTB3), constraints: attack=70%, each>=5% ===
Ann.Return=0.2277, Ann.Vol=0.2073, Sharpe=0.9909
Attack wei